<!-- @format -->

# 03 - Baseline Models

Notebook này huấn luyện và so sánh các mô hình baseline: **Linear Regression** và **Random Forest**.

**Dữ liệu đầu vào:** `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`.


<!-- @format -->

## 1. Setup & Load Data


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    X_TRAIN_FILE, X_TEST_FILE, Y_TRAIN_FILE, Y_TEST_FILE,
    TARGET_COL, RANDOM_STATE, MODELS_DIR
)
from src.utils import save_model

sns.set_theme(style="whitegrid")
print("Setup OK")

In [ ]:
# Đọc dữ liệu đã tách sẵn
X_train = pd.read_csv(X_TRAIN_FILE)
X_test = pd.read_csv(X_TEST_FILE)
y_train = pd.read_csv(Y_TRAIN_FILE)[TARGET_COL]
y_test = pd.read_csv(Y_TEST_FILE)[TARGET_COL]

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")
print(f"\nCác cột features ({X_train.shape[1]}):")
print(list(X_train.columns))

<!-- @format -->

## 2. Hàm đánh giá chung

Mỗi model sẽ được đánh giá bằng 3 chỉ số:

- **MAE** (Mean Absolute Error): Sai số tuyệt đối trung bình
- **RMSE** (Root Mean Squared Error): Sai số bình phương trung bình gốc
- **R²** (Coefficient of Determination): Mức độ giải thích của mô hình (càng gần 1 càng tốt)


In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    """Train model, dự đoán và tính metrics."""
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model),
    ])
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)
    
    mae = mean_absolute_error(y_te, y_pred)
    rmse = mean_squared_error(y_te, y_pred) ** 0.5
    r2 = r2_score(y_te, y_pred)
    
    print(f"{'='*40}")
    print(f"Model: {name}")
    print(f"  MAE  = {mae:,.2f}")
    print(f"  RMSE = {rmse:,.2f}")
    print(f"  R²   = {r2:.4f}")
    
    return pipe, {"model": name, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "R2": round(r2, 4)}

<!-- @format -->

## 3. Train Linear Regression

Mô hình đơn giản nhất - tìm quan hệ tuyến tính giữa các features và giá vé.


In [ ]:
results = []

lr_model = LinearRegression()
lr_pipe, lr_metrics = evaluate_model("Linear Regression", lr_model, X_train, y_train, X_test, y_test)
results.append(lr_metrics)

<!-- @format -->

## 4. Train Decision Tree

Mô hình cây quyết định - có thể nắm bắt quan hệ phi tuyến nhưng dễ bị overfit.


In [ ]:
dt_model = DecisionTreeRegressor(random_state=RANDOM_STATE)
dt_pipe, dt_metrics = evaluate_model("Decision Tree", dt_model, X_train, y_train, X_test, y_test)
results.append(dt_metrics)

<!-- @format -->

## 5. Train Random Forest

Mô hình ensemble sử dụng nhiều cây quyết định - thường cho kết quả tốt hơn Decision Tree đơn lẻ.


In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf_pipe, rf_metrics = evaluate_model("Random Forest", rf_model, X_train, y_train, X_test, y_test)
results.append(rf_metrics)

<!-- @format -->

## 6. So sánh các mô hình Baseline


In [ ]:
# Bảng so sánh
results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
display(results_df)

# Biểu đồ so sánh
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_list = ["MAE", "RMSE", "R2"]
colors = ["#e74c3c", "#f39c12", "#2ecc71"]

for ax, metric, color in zip(axes, metrics_list, colors):
    df_sorted = results_df.sort_values(metric, ascending=(metric != "R2"))
    ax.barh(df_sorted["model"], df_sorted[metric], color=color, edgecolor="white")
    ax.set_title(metric, fontsize=14, fontweight="bold")
    for i, v in enumerate(df_sorted[metric]):
        ax.text(v, i, f" {v}", va="center", fontsize=10)

plt.suptitle("So sánh Baseline Models", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

<!-- @format -->

## 7. Lưu mô hình Baseline


In [ ]:
# Lưu mô hình baseline
models_to_save = {
    "linear_regression": lr_pipe,
    "decision_tree": dt_pipe,
    "random_forest": rf_pipe,
}
for name, pipe in models_to_save.items():
    path = MODELS_DIR / "baseline" / f"{name}.joblib"
    save_model(pipe, path)
    print(f"Saved: {path}")

<!-- @format -->

## Nhận xét

- **Random Forest** cho kết quả tốt nhất với R² cao nhất và RMSE thấp nhất.
- **Decision Tree** khá tốt nhưng kém hơn RF do dễ bị overfit trên dữ liệu training.
- **Linear Regression** cho kết quả thấp nhất vì giá vé có quan hệ **phi tuyến** với các đặc trưng.

**Bước tiếp theo:** Notebook 04 sẽ thử tuning Random Forest và thêm XGBoost để cải thiện kết quả.
